# Bootstrap Yield Curve — Package Demo

A live walkthrough of the `yield_curve` package: bootstrapping Treasury discount factors three ways, fitting a Nelson-Siegel curve, and running a Markowitz portfolio analysis on five large-cap tech stocks.

> **Original exercise:** August 2022 (see `notebooks/Bootstrapping_Yield_Curve.ipynb`).
> **Package surface:** September 2026.
> **Run:** `pip install -e ".[dev,yfinance]"` then open this notebook.

In [ ]:
# --- Imports ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from yield_curve import YieldCurve, NelsonSiegel, PortfolioAnalyzer, fetch_prices_from_yfinance

sns.set_theme(style="whitegrid", palette="dark")
np.set_printoptions(suppress=True, formatter={'float_kind': '{:0.4f}'.format})

## 1  Treasury yield curve — bootstrapping three ways

We start with the 10 Treasury bonds from the original exercise. Each row is `(maturity_years, dirty_price, annual_coupon_rate)` at par value 100. The `YieldCurve` class builds the cash-flows matrix and bootstraps discount factors.

In [ ]:
# 10 Treasury bonds (maturity, price, coupon) from the original project
maturities = np.arange(1, 11)
prices     = np.array([96.60, 93.71, 91.56, 90.24, 89.74, 90.04, 91.09, 92.82, 95.19, 98.14])
coupons    = np.linspace(0.015, 0.0375, num=10)

bonds = np.column_stack((maturities, prices, coupons))
bonds

In [ ]:
yc = YieldCurve(bonds)

print('Discount factors — Matrix operations:')
print(yc.discount_factors('Matrix operations'))
print()
print('Discount factors — Global Solver:')
print(yc.discount_factors('Global Solver'))
print()
print('Discount factors — Iterative procedure:')
print(yc.discount_factors('Iterative Procedure'))

**Sanity check:** all three methods should give the same discount factors (to within numerical tolerance). The matrix method is exact; the solver and iterative procedure are alternative routes to the same answer.

In [ ]:
df_m   = yc.discount_factors('Matrix operations')
df_s   = yc.discount_factors('Global Solver')
df_i   = yc.discount_factors('Iterative Procedure')

print('Max abs diff (matrix vs solver):', np.max(np.abs(df_m - df_s)))
print('Max abs diff (matrix vs iterative):', np.max(np.abs(df_m - df_i)))

# Reproduce market prices from the discount factors
print()
print('Prices recovered from cash flows @ DF (should match market):')
print(yc.cash_flows @ df_m)

## 2  Spot rates, YTM, and forward rates

From the discount factors we derive the three standard rate views.

In [ ]:
spot  = yc.spot_rates()
ytm   = yc.bonds_ytm()
fwd   = yc.forward_rates()

print('Spot rates (%):', np.round(100 * spot, 2))
print('YTM (%):       ', np.round(100 * ytm, 2))
print('1y Forward (%):', np.round(100 * fwd, 2))

In [ ]:
yc.plot_rates(
    title='US Treasury Yield Curve — Spot, YTM, and Forward Rates',
    save_path='yield_curve_overview.png',
    show=True
)

## 3  Nelson-Siegel parametric fit

The bootstrapped curve from 10 bonds is jagged. Nelson-Siegel smooths it into a parametric form with four parameters: level $\beta_0$, slope $\beta_1$, curvature $\beta_2$, and decay $\tau$:

$$f(t) = \beta_0 + \beta_1 e^{-t/\tau} + \beta_2 \frac{t}{\tau} e^{-t/\tau}$$

We fit it to the bootstrapped discount factors by least squares.

In [ ]:
ns = NelsonSiegel(
    maturities=bonds[:, 0].astype(float),
    discount_factors=yc.discount_factors('Matrix operations'),
)
ns.fit()

beta0, beta1, beta2, tau = ns.params
print('Fitted Nelson-Siegel parameters:')
print(f'  beta0 (level)     = {beta0:.6f}')
print(f'  beta1 (slope)     = {beta1:.6f}')
print(f'  beta2 (curvature) = {beta2:.6f}')
print(f'  tau (decay)       = {tau:.4f}')

In [ ]:
ns.plot(
    title='Nelson-Siegel fit vs bootstrapped discount factors',
    save_path='nelson_siegel_fit.png',
    show=True
)

## 4  Portfolio analysis — five tech stocks

Now the equity side: daily price panel for AAPL, IBM, MSFT, GOOG, AMZN. We fetch live data from yfinance (or use a cached CSV), then compute returns, the correlation matrix, and the Markowitz efficient frontier.

> If yfinance is not installed, the notebook falls back to generating synthetic data for the demo.

In [ ]:
tickers = ['AAPL', 'IBM', 'MSFT', 'GOOG', 'AMZN']

try:
    prices = fetch_prices_from_yfinance(tickers, start='2019-01-01')
    print(f'Fetched {len(prices)} daily observations for {len(tickers)} tickers')
    print('Date range:', prices.index.min().date(), 'to', prices.index.max().date())
except Exception as e:
    print(f'yfinance fetch failed ({e}) — using synthetic data for the demo')
    np.random.seed(42)
    dates = pd.bdate_range('2019-01-01', periods=500)
    base = np.random.randn(500, 5).cumsum(axis=0) * 0.02 + 100
    prices = pd.DataFrame(base, index=dates, columns=tickers)
    prices.iloc[:, 1] *= 1.2  # IBM higher base price
    prices.iloc[:, 3] *= 0.1  # GOOG lower base price


In [ ]:
prices.head()

In [ ]:
# Build the analyzer — risk-free rate can come from the bootstrapped curve or an external source
risk_free = float(spot[0])  # 1-year spot rate as a proxy, or set manually: risk_free = 0.045
pa = PortfolioAnalyzer(prices, risk_free_rate=risk_free)

print(f'Risk-free rate used: {risk_free:.4f} ({100*risk_free:.2f}%)')
print(f'Observation period: {len(pa.returns)} trading days')
print(f'Annualized mean returns:')
print(pa.mean_returns.sort_values(ascending=False))

In [ ]:
pa.cumulative_returns_plot(
    title=f'Cumulative returns ({prices.index.min().date()} to {prices.index.max().date()})',
    save_path='cumulative_returns.png',
    show=True
)

In [ ]:
pa.correlation_heatmap(
    save_path='correlation_heatmap.png',
    show=True
)

In [ ]:
frontier = pa.efficient_frontier(n_points=100)
max_sharpe = pa.max_sharpe_portfolio()

print('Maximum Sharpe portfolio:')
for ticker, w in zip(tickers, max_sharpe['weights']):
    if w > 1e-6:
        print(f'  {ticker:6s}: {w*100:6.2f}%')
print(f'  Volatility : {max_sharpe["volatility"]*100:.2f}%')
print(f'  Return     : {max_sharpe["return"]*100:.2f}%')
print(f'  Sharpe     : {max_sharpe["sharpe"]:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(frontier['volatility']*100, frontier['return']*100, 'b-', alpha=0.5, label='Efficient frontier')
ax.scatter(
    [max_sharpe['volatility']*100],
    [max_sharpe['return']*100],
    color='red', s=150, zorder=5, label=f'Max Sharpe ({max_sharpe["sharpe"]:.2f})'
)
for ticker, ret, vol in zip(tickers, pa.mean_returns, np.sqrt(np.diag(pa.cov_matrix))):
    ax.scatter(vol*100, ret*100, alpha=0.6, label=ticker)

ax.set_xlabel('Annualized volatility (%)')
ax.set_ylabel('Annualized return (%)')
ax.set_title('Markowitz Efficient Frontier — Tech Portfolio')
ax.legend()
ax.grid(alpha=0.3)
fig.savefig('efficient_frontier.png', dpi=150, bbox_inches='tight')
plt.show()

## 5  What this repo demonstrates

A hiring manager looking at this repo should see:

- **Quant finance fundamentals:** bootstrapping, spot/forward rates, Nelson-Siegel — all implemented, not just called from a library.
- **Python package hygiene:** `src/` layout, `pyproject.toml`, type-aware code, `pytest` suite, GitHub Actions CI on Python 3.9–3.12.
- **Data pipeline:** fetching from an external API (yfinance), cleaning, analyzing, visualizing, saving artifacts.
- **Portfolio theory:** mean-variance optimization, efficient frontier, max-Sharpe tangency portfolio — the bread-and-butter of quant / data-science finance roles.
- **Original work preserved:** the 2022 notebook is untouched in `notebooks/`, showing the evolution from exploration to engineered package.

### Next steps (open issues)

- Nelson-Siegel-Svensson (three-factor extension)
- Cubic / monotonic spline bootstrapping
- FRED / Treasury.gov data connectors
- Arbitrage-free / monotonicity unit tests on the fitted curve